## Анализ данных в Python

*Алла Тамбовцева, Ксения Зайцева*

### Практикум 2. Датафреймы pandas: добавление новых столбцов и формат дата-время

* Добавление новых столбцов на основе старых
* Метод `.apply()` и lambda-функции
* Создание бинарных столбцов и метод `.astype()`
* Методы `.isin()` и `.str.contains()`
* Тип `datetime` в `pandas` и атрибуты даты-времени

С сайта [nplus1.ru](https://nplus1.ru/), где публикуются новости науки, были выгружены данные по статьям за 2026 год.

Переменные в файле `nplus_one.csv`:

* `title`: заголовок новости;		
* `author`: автор;
* `date`: дата публикации;
* `diffc`: сложность новости;
* `rubrics`: рубрики (тематические разделы);
* `text`: полный текст новости.

Загрузите данные из файла и выполните задания ниже.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("nplus_one.csv")

### Задача 1

Добавьте столбец `diffc_perc` с относительной сложностью статей (в процентах относительно максимального значения).

In [3]:
df["diffc_perc"] = df["diffc"] / df["diffc"].max() * 100
df["diffc_perc"].describe()

count    3706.000000
mean       29.513471
std        13.239570
min        10.989011
25%        20.879121
50%        26.373626
75%        35.164835
max       100.000000
Name: diffc_perc, dtype: float64

### Задача 2

Добавьте столбец `title_len` с длиной заголовков – количеством символов в них.

In [4]:
# вариант 1: метод .apply() + len()

df["title_len"] = df["title"].apply(len)
df["title_len"].describe()

count    3706.000000
mean      116.737992
std        23.346990
min        44.000000
25%       101.000000
50%       116.000000
75%       132.000000
max       228.000000
Name: title_len, dtype: float64

In [5]:
# вариант 2: метод .len() внутри str

df["title_len"] = df["title"].str.len()
df["title_len"].describe()

count    3706.000000
mean      116.737992
std        23.346990
min        44.000000
25%       101.000000
50%       116.000000
75%       132.000000
max       228.000000
Name: title_len, dtype: float64

### Задача 3

Добавьте столбец `n_rubs` с количеством рубрик, к которым относится каждая новость.

In [6]:
# пишем lambda-функцию и растягиваем через .apply()

df["n_rubs"] = df["rubrics"].apply(lambda x: len(x.split(",")))
df["n_rubs"].value_counts()

n_rubs
1    2556
2    1005
3     137
4       8
Name: count, dtype: int64

### Задача 4

Добавьте столбец `diffc_ord` с закодированной сложностью статьи: 

* сложность выше 3 – значение "high",
* сложность от 2 до 3 – значение "medium",
* сложность ниже 2 – значение "low".

In [7]:
# тут запасная ветка с x > 0,
# теоретически на случай наличия пропусков,
# чтобы не ломалось при apply()

def my_encode(x):
    if x > 3:
        y = "high"
    elif x > 2:
        y = "medium"
    elif x > 0:
        y = "low"
    else:
        y = x
    return y

df["diffc_ord"] = df["diffc"].apply(my_encode)
df["diffc_ord"].value_counts()

diffc_ord
medium    1296
high      1264
low       1146
Name: count, dtype: int64

### Задача 5

Получите список уникальных значений столбца `rubrics`, отсортированный по алфавиту.

In [8]:
r = sorted(df["rubrics"].unique())
print(r)

['IT', 'IT, Биология', 'IT, Биология, Мезонин', 'IT, Гаджеты', 'IT, Лингвистика', 'IT, Лингвистика, История', 'IT, Математика', 'IT, Медицина', 'IT, Мезонин', 'IT, Наука', 'IT, Психология', 'IT, Роботы и дроны', 'IT, Связь', 'IT, Физика, Космонавтика', 'Авиация', 'Авиация, Мезонин', 'Авиация, Транспорт', 'Антропология', 'Антропология, Археология', 'Антропология, Археология, Генетика', 'Антропология, Археология, История', 'Антропология, Генетика, Археология', 'Антропология, Генетика, История, Археология', 'Антропология, История', 'Антропология, История, Археология', 'Антропология, Медицина', 'Антропология, Медицина, Археология', 'Антропология, Палеонтология', 'Археология', 'Археология, Антропология', 'Археология, Антропология, История', 'Археология, Биология', 'Археология, Ботаника', 'Археология, Генетика, Антропология', 'Археология, Геология', 'Археология, Геология, История', 'Археология, Зоология', 'Археология, Зоология, Генетика', 'Археология, История', 'Археология, История, Лингвист

### Задача 6

Добавьте бинарный столбец `cosmos1`, где значение 1 соответствует рубрикам `Космос` и `Астрономия` (точное соответствие, рубрика из одного слова).

In [9]:
# вариант 1: оператор ИЛИ

df["cosmos1"] = ((df["rubrics"] == "Астрономия") | (df["rubrics"] == "Космос")).astype(int)
df["cosmos1"].value_counts()

cosmos1
0    3553
1     153
Name: count, dtype: int64

In [10]:
# вариант 2: метод .isin()

df["cosmos1"] = df["rubrics"].isin(["Астрономия", "Космос"]).astype(int)
df["cosmos1"].value_counts()

cosmos1
0    3553
1     153
Name: count, dtype: int64

### Задача 7

Добавьте бинарный столбец `cosmos2`, где значение 1 соответствует рубрикам `Космос` и `Астрономия` (частичное соответствие, слово входит в перечень рубрик).

In [11]:
# вариант 1: метод .contains() внутри str + оператор ИЛИ

df["cosmos2"] = ((df["rubrics"].str.contains("Астрономия")) | 
                 (df["rubrics"].str.contains("Космос"))).astype(int)
df["cosmos2"].value_counts()

cosmos2
0    3520
1     186
Name: count, dtype: int64

In [12]:
# вариант 2: метод .contains() внутри str + оператор ИЛИ внутри кавычек,
# в .contains() поддерживаются регулярные выражения, в них есть свой опеартор |

df["cosmos2"] = df["rubrics"].str.contains("Астрономия|Космос").astype(int)
df["cosmos2"].value_counts()

cosmos2
0    3520
1     186
Name: count, dtype: int64

In [13]:
# вариант 3: если забыли про str.contains(),
# пишем функцию с обычным in и применяем

df["cosmos2"] = df["rubrics"].apply(lambda x: "Астрономия" in x or "Космос" in x)
df["cosmos2"].value_counts()

cosmos2
False    3520
True      186
Name: count, dtype: int64

### Задача 8

Приведите столбец с датой публикацией к типу *дата-время*. Извлеките год, месяц, день публикации новости и день недели и сохраните их столбцы `year`, `month`, `day` и `weekday`. 

Подсказка: функция `.to_datetime()` из `pandas`, а затем – разные [атрибуты](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.html#pandas.DatetimeIndex) с префиксом `dt`.

In [14]:
# изначально тип столбца – строка
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 3706 entries, 0 to 3705
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  3706 non-null   int64  
 1   title       3706 non-null   str    
 2   author      3706 non-null   str    
 3   date        3706 non-null   str    
 4   diffc       3706 non-null   float64
 5   rubrics     3706 non-null   str    
 6   text        3706 non-null   str    
 7   diffc_perc  3706 non-null   float64
 8   title_len   3706 non-null   int64  
 9   n_rubs      3706 non-null   int64  
 10  diffc_ord   3706 non-null   str    
 11  cosmos1     3706 non-null   int64  
 12  cosmos2     3706 non-null   bool   
dtypes: bool(1), float64(2), int64(4), str(6)
memory usage: 21.9 MB
None


In [15]:
# переделываем, теперь тип datetime

df["date"] = pd.to_datetime(df["date"])
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 3706 entries, 0 to 3705
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  3706 non-null   int64         
 1   title       3706 non-null   str           
 2   author      3706 non-null   str           
 3   date        3706 non-null   datetime64[us]
 4   diffc       3706 non-null   float64       
 5   rubrics     3706 non-null   str           
 6   text        3706 non-null   str           
 7   diffc_perc  3706 non-null   float64       
 8   title_len   3706 non-null   int64         
 9   n_rubs      3706 non-null   int64         
 10  diffc_ord   3706 non-null   str           
 11  cosmos1     3706 non-null   int64         
 12  cosmos2     3706 non-null   bool          
dtypes: bool(1), datetime64[us](1), float64(2), int64(4), str(5)
memory usage: 21.9 MB
None


In [16]:
# извлекаем атрибуты из dt (year, month, day)
# а потом оттуда же метод day_name()

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.day_name()

In [17]:
# можем забрать день недели в виде числа, 
# 0 – Monday, 6 - Sunday

df["date"].dt.day_of_week

0       4
1       4
2       4
3       0
4       0
       ..
3701    3
3702    3
3703    3
3704    3
3705    3
Name: date, Length: 3706, dtype: int32

In [18]:
# можем забрать день года,
# от 1 до 365

df["date"].dt.day_of_year

0         3
1         3
2         3
3         6
4         6
       ... 
3701    316
3702    316
3703    316
3704    316
3705    316
Name: date, Length: 3706, dtype: int32

In [19]:
# можем забрать время,
# раз здесь его не было, только дата, автоматически будет полночь

df["date"].dt.time

0       00:00:00
1       00:00:00
2       00:00:00
3       00:00:00
4       00:00:00
          ...   
3701    00:00:00
3702    00:00:00
3703    00:00:00
3704    00:00:00
3705    00:00:00
Name: date, Length: 3706, dtype: object